# LightFM baseline для контеста по рекомендациям

> **Цель проекта:** построить воспроизводимый implicit-feedback baseline для ранжирования пар `user × track`.

В ноутбуке последовательно выполнены:

1. загрузка и проверка исходных данных;
2. агрегация повторных прослушиваний пользователя;
3. разведочный анализ и выбор порога положительного взаимодействия;
4. воспроизводимое разбиение на train/validation без утечки таргета;
5. обучение **LightFM** с loss `BPR`;
6. подбор гиперпараметров через **Optuna** по `NDCG@10`;
7. переобучение лучшей модели на всех доступных взаимодействиях;
8. генерация файла submission в формате контеста.

### Важные методологические решения

- `time` трактуется как implicit signal: чем дольше трек слушали, тем выше вес взаимодействия.
- Несколько строк для одной пары `user × track` агрегируются до одной записи.
- Для тюнинга используется только validation; финальная модель после выбора параметров обучается на всех наблюдаемых взаимодействиях.
- Все случайные операции используют фиксированный `SEED`, поэтому ноутбук можно воспроизвести.
- Метрика считается только по пользователям и объектам, присутствующим в train. Это стандартное ограничение для warm-start baseline и явно учитывается при формировании submission.

> **Ожидаемая структура репозитория**
> ```text
> project/
> ├── data/contest_1/raw_data/train.csv
> ├── data/contest_1/raw_data/test.csv
> └── notebooks/lightfm_baseline.ipynb
> ```

## 1. Импорты и конфигурация

Если окружение ещё не настроено, зависимости можно установить одной командой:

```bash
pip install pandas numpy scipy matplotlib seaborn lightfm optuna tqdm
```

Для GitHub ноутбук намеренно не запускает установку пакетов автоматически: это делает выполнение предсказуемым в CI и локальных окружениях.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lightfm import LightFM
from lightfm.data import Dataset

import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 31337
N_THREADS = 4
TOP_K = 10
VALID_SIZE = 0.20
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 3
POSITIVE_THRESHOLD = 0.75
N_TRIALS = 30

DATA_DIR = Path("data/contest_1/raw_data")
OUTPUT_DIR = Path("data/contest_1/processed")
TRAIN_PATH = DATA_DIR / "train.csv"
SUBMISSION_INPUT_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = OUTPUT_DIR / "submit_lightfm_bpr.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="notebook")

print(f"Seed: {SEED}")
print(f"Train path: {TRAIN_PATH.resolve()}")
print(f"Submission path: {SUBMISSION_PATH.resolve()}")

## 2. Загрузка и первичная проверка данных

В исходном `train.csv` ожидаются три столбца:

- `user` — идентификатор пользователя;
- `track` — идентификатор трека;
- `time` — величина наблюдаемого взаимодействия.

Проверки ниже специально выполняются до любой агрегации: так ошибки в формате данных не маскируются последующей обработкой.

In [ ]:
def validate_interactions(frame: pd.DataFrame, required_columns: set[str]) -> None:
    missing = required_columns.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if frame.empty:
        raise ValueError("The interaction table is empty.")
    if frame[list(required_columns)].isna().any().any():
        raise ValueError("Required columns contain missing values.")
    if (frame["time"] < 0).any():
        raise ValueError("Column 'time' must be non-negative.")

raw = pd.read_csv(TRAIN_PATH)
validate_interactions(raw, {"user", "track", "time"})
raw = raw[["user", "track", "time"]].copy()
raw["user"] = raw["user"].astype(str)
raw["track"] = raw["track"].astype(str)
raw["time"] = pd.to_numeric(raw["time"], errors="raise").astype("float32")

print(f"Rows: {len(raw):,}")
print(f"Users: {raw['user'].nunique():,}")
print(f"Tracks: {raw['track'].nunique():,}")
raw.head()

## 3. Агрегация взаимодействий и выбор положительных сигналов

Одна пара может встречаться в исходном логе несколько раз. Сумма `time` сохраняет интенсивность взаимодействия, а `count` помогает контролировать качество агрегации. В baseline оставляем пары с `total_time > 0.75` — это порог из исходного решения, теперь вынесенный в именованную константу.

In [ ]:
interactions = (
    raw.groupby(["user", "track"], as_index=False)
       .agg(total_time=("time", "sum"),
            play_count=("time", "size"))
)

interactions = interactions.loc[
    interactions["total_time"] > POSITIVE_THRESHOLD
].copy()
interactions["log_time"] = np.log1p(interactions["total_time"]).astype("float32")

print(f"Unique positive interactions: {len(interactions):,}")
print(f"Users after threshold: {interactions['user'].nunique():,}")
print(f"Tracks after threshold: {interactions['track'].nunique():,}")
interactions.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(interactions["total_time"], bins=50, kde=True, ax=axes[0], color="#4472C4")
axes[0].set_title("Распределение total_time")
axes[0].set_xlabel("Суммарное время")
axes[0].set_ylabel("Число пар")

sns.histplot(interactions["log_time"], bins=50, kde=True, ax=axes[1], color="#ED7D31")
axes[1].set_title("log1p(total_time)")
axes[1].set_xlabel("Логарифмированный вес")
axes[1].set_ylabel("Число пар")

plt.tight_layout()
plt.show()

interactions["total_time"].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]).to_frame()

## 4. Воспроизводимое train/validation разбиение

Сначала разделяем агрегированные пары по фиксированной случайной маске. Затем определяем warm-start множество пользователей и треков **только по train**. Это предотвращает использование информации о validation при построении словаря объектов.

Ограничения `MIN_USER_INTERACTIONS` и `MIN_ITEM_INTERACTIONS` нужны для того, чтобы в train у пользователя и трека было достаточно наблюдений для обучения и проверки. Пользователи/треки, не прошедшие фильтр, не удаляются молча из исходной таблицы: их количество будет показано в отчёте.

In [ ]:
split_rng = np.random.default_rng(SEED)
interactions["is_validation"] = split_rng.random(len(interactions)) < VALID_SIZE

train_candidates = interactions.loc[~interactions["is_validation"]].copy()
valid_candidates = interactions.loc[interactions["is_validation"]].copy()

user_counts = train_candidates["user"].value_counts()
item_counts = train_candidates["track"].value_counts()
known_users = set(user_counts[user_counts >= MIN_USER_INTERACTIONS].index)
known_tracks = set(item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index)

train = train_candidates.loc[
    train_candidates["user"].isin(known_users)
    & train_candidates["track"].isin(known_tracks)
].copy()
validation = valid_candidates.loc[
    valid_candidates["user"].isin(known_users)
    & valid_candidates["track"].isin(known_tracks)
].copy()

# В validation оставляем только пользователей, которые имеют хотя бы один target.
validation = validation.loc[validation["user"].isin(set(train["user"]))].copy()

split_report = pd.DataFrame({
    "rows": [len(interactions), len(train_candidates), len(valid_candidates), len(train), len(validation)],
    "users": [interactions["user"].nunique(), train_candidates["user"].nunique(), valid_candidates["user"].nunique(), train["user"].nunique(), validation["user"].nunique()],
    "tracks": [interactions["track"].nunique(), train_candidates["track"].nunique(), valid_candidates["track"].nunique(), train["track"].nunique(), validation["track"].nunique()],
}, index=["all_positive", "train_candidates", "validation_candidates", "train_warm_start", "validation_warm_start"])
split_report

## 5. Кодирование данных в формат LightFM

`Dataset` строит стабильное отображение исходных идентификаторов в индексы матриц. В качестве `sample_weight` передаём `log1p(total_time)`: редкие короткие взаимодействия сохраняются как положительные, но длительные прослушивания получают больший вес.

In [ ]:
dataset = Dataset()
dataset.fit(
    users=sorted(known_users),
    items=sorted(known_tracks),
)

user_id_map, _, item_id_map, _ = dataset.mapping()

def build_lightfm_interactions(frame: pd.DataFrame, with_weights: bool = True):
    tuples = frame[["user", "track", "log_time"]].itertuples(
        index=False, name=None
    )
    return dataset.build_interactions(tuples)

train_interactions, train_weights = build_lightfm_interactions(train)
validation_interactions, _ = build_lightfm_interactions(validation)

print(f"Interaction matrix shape: {train_interactions.shape}")
print(f"Train non-zero interactions: {train_interactions.nnz:,}")
print(f"Validation non-zero interactions: {validation_interactions.nnz:,}")

## 6. Метрики ранжирования

Для implicit recommendation важнее порядок объектов, чем абсолютная величина score. Поэтому используем:

- `precision@10` — доля релевантных рекомендаций в top-10;
- `recall@10` — доля найденных релевантных объектов;
- `NDCG@10` — основная метрика тюнинга, учитывающая позицию релевантного объекта.

При расчёте рангов train-взаимодействия исключаются из кандидатов, чтобы модель не получала бонус за уже известные пользователю треки.

In [ ]:
def ndcg_at_k(model, test_interactions, train_interactions, k=10, num_threads=4):
    ranks = model.predict_rank(
        test_interactions,
        train_interactions=train_interactions,
        num_threads=num_threads,
    ).tocsr()

    scores = []
    for user_idx in range(ranks.shape[0]):
        user_ranks = ranks.data[ranks.indptr[user_idx]: ranks.indptr[user_idx + 1]]
        if user_ranks.size == 0:
            continue
        user_ranks = np.sort(user_ranks)[:k]
        discounts = 1.0 / np.log2(user_ranks + 2.0)
        dcg = discounts.sum()
        ideal = np.arange(len(user_ranks), dtype=float)
        idcg = (1.0 / np.log2(ideal + 2.0)).sum()
        scores.append(float(dcg / idcg) if idcg else 0.0)

    return float(np.mean(scores)) if scores else 0.0


def evaluate_model(model, test_interactions, train_interactions, k=10):
    from lightfm.evaluation import precision_at_k, recall_at_k
    return {
        "precision@10": float(np.mean(precision_at_k(model, test_interactions, train_interactions=train_interactions, k=k, num_threads=N_THREADS))),
        "recall@10": float(np.mean(recall_at_k(model, test_interactions, train_interactions=train_interactions, k=k, num_threads=N_THREADS))),
        "ndcg@10": ndcg_at_k(model, test_interactions, train_interactions, k=k, num_threads=N_THREADS),
    }

## 7. Подбор гиперпараметров

Оптимизируем только параметры LightFM, а validation используем исключительно для сравнения trial-ов. Для экономии времени число trial-ов вынесено в `N_TRIALS`; перед публикацией можно увеличить его, не меняя остальной pipeline.

In [ ]:
def make_model(params: dict) -> LightFM:
    return LightFM(
        loss=params["loss"],
        no_components=params["no_components"],
        learning_rate=params["learning_rate"],
        user_alpha=params["user_alpha"],
        item_alpha=params["item_alpha"],
        max_sampled=params["max_sampled"],
        random_state=SEED,
    )


def objective(trial: optuna.Trial) -> float:
    params = {
        "loss": trial.suggest_categorical("loss", ["bpr", "warp"]),
        "no_components": trial.suggest_int("no_components", 16, 64),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 1e-1, log=True),
        "epochs": trial.suggest_int("epochs", 15, 40),
        "user_alpha": trial.suggest_float("user_alpha", 1e-6, 1e-2, log=True),
        "item_alpha": trial.suggest_float("item_alpha", 1e-6, 1e-2, log=True),
        "max_sampled": trial.suggest_int("max_sampled", 20, 100),
    }

    model = make_model(params)
    model.fit(
        train_interactions,
        sample_weight=train_weights,
        epochs=params["epochs"],
        num_threads=N_THREADS,
        verbose=False,
    )
    score = ndcg_at_k(model, validation_interactions, train_interactions, k=TOP_K, num_threads=N_THREADS)
    trial.set_user_attr("validation_ndcg@10", score)
    return score

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

best_params = study.best_trial.params
best_score = study.best_value
print(f"Best validation NDCG@{TOP_K}: {best_score:.5f}")
best_params

In [ ]:
trials = study.trials_dataframe()
trials = trials.sort_values("value", ascending=False)
trials[["number", "value", "params_loss", "params_no_components", "params_learning_rate", "params_epochs"]].head(10)

## 8. Финальное обучение и контрольная оценка

После выбора гиперпараметров обучаем модель на полном наборе warm-start взаимодействий (`train + validation`). Значение validation выше остаётся честной оценкой этапа выбора параметров; после переобучения отдельной unbiased оценки уже нет, поэтому результат не выдаём за финальную метрику на скрытом контестном тесте.

In [ ]:
all_warm_interactions = pd.concat([train, validation], ignore_index=True)
final_dataset = Dataset()
final_dataset.fit(
    users=sorted(all_warm_interactions["user"].unique()),
    items=sorted(all_warm_interactions["track"].unique()),
)

final_interactions, final_weights = final_dataset.build_interactions(
    all_warm_interactions[["user", "track", "log_time"]].itertuples(index=False, name=None)
)

final_model = make_model(best_params)
final_model.fit(
    final_interactions,
    sample_weight=final_weights,
    epochs=best_params["epochs"],
    num_threads=N_THREADS,
    verbose=True,
)

print(f"Final matrix shape: {final_interactions.shape}")
print(f"Final matrix nnz: {final_interactions.nnz:,}")

## 9. Генерация submission

Для каждой строки `test.csv` сохраняем исходную пару `user × track` и добавляем score модели. Известные пользователи и треки получают LightFM score. Для cold-start строк используем детерминированный popularity fallback: это лучше, чем случайный шум, и сохраняет воспроизводимость.

Если контест ожидает другой формат файла, достаточно изменить список колонок в последней ячейке.

In [ ]:
submission_input = pd.read_csv(SUBMISSION_INPUT_PATH)
validate_interactions(
    submission_input.assign(time=0.0),
    {"user", "track"},
)
submission_input = submission_input[["user", "track"]].copy()
submission_input["user"] = submission_input["user"].astype(str)
submission_input["track"] = submission_input["track"].astype(str)

final_user_map, _, final_item_map, _ = final_dataset.mapping()
known_user_mask = submission_input["user"].isin(final_user_map)
known_item_mask = submission_input["track"].isin(final_item_map)
known_pair_mask = known_user_mask & known_item_mask

scores = np.full(len(submission_input), np.nan, dtype="float32")
if known_pair_mask.any():
    user_indices = submission_input.loc[known_pair_mask, "user"].map(final_user_map).to_numpy()
    item_indices = submission_input.loc[known_pair_mask, "track"].map(final_item_map).to_numpy()
    scores[known_pair_mask.to_numpy()] = final_model.predict(
        user_indices,
        item_indices,
        num_threads=N_THREADS,
    )

# Deterministic fallback for cold-start users/items.
item_popularity = all_warm_interactions["track"].value_counts(normalize=True)
fallback = submission_input["track"].map(item_popularity).fillna(0.0).to_numpy(dtype="float32")
scores = np.where(np.isnan(scores), fallback, scores)

submission = submission_input.copy()
submission["score"] = scores
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"Rows written: {len(submission):,}")
print(f"Known user-track pairs: {known_pair_mask.mean():.1%}")
print(f"Saved to: {SUBMISSION_PATH}")
submission.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(submission["score"], bins=50, ax=ax, color="#70AD47")
ax.set_title("Распределение scores в submission")
ax.set_xlabel("Model score / popularity fallback")
ax.set_ylabel("Число строк")
plt.tight_layout()
plt.show()

## 10. Итоги и ограничения

### Что получено

- воспроизводимый LightFM baseline с BPR/WARP search;
- прозрачный validation protocol и `NDCG@10` как objective;
- финальная модель, обученная на всех доступных warm-start взаимодействиях;
- submission-файл, сохраняющий исходный порядок строк теста.

### Ограничения baseline

1. В данных нет времени события, поэтому используется случайное, а не temporal split.
2. Нет контентных признаков треков или пользователей, поэтому модель не решает cold-start полностью.
3. Validation score нельзя напрямую сравнивать с публичным leaderboard без фиксации протокола контеста.
4. Для улучшения качества следующим шагом стоит проверить temporal split, popularity baseline, item/user features и ансамбль с implicit ALS/BPR.

**Результат:** `data/contest_1/processed/submit_lightfm_bpr.csv`.